<a href="https://colab.research.google.com/github/gowdarafe-beep/namratha/blob/main/Copy_of_aidigitaltwin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai

import os
import json
import sqlite3
from datetime import datetime
from getpass import getpass
from google import genai

# ============================================================
# AI DIGITAL TWIN
# ============================================================

print("=" * 65)
print("          AI DIGITAL TWIN OF A PERSON")
print("=" * 65)

# ------------------------------------------------------------
# 1. GEMINI API SETUP
# ------------------------------------------------------------

API_KEY = getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"


def ask_ai(prompt):
    """Send a prompt to Gemini and return the response."""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text


# ------------------------------------------------------------
# 2. DATABASE
# ------------------------------------------------------------

DB_NAME = "digital_twin.db"

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS profile (
    id INTEGER PRIMARY KEY,
    name TEXT,
    interests TEXT,
    skills TEXT,
    career_goal TEXT,
    learning_style TEXT,
    preferences TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS memories (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    memory TEXT,
    category TEXT,
    created_at TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS goals (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    goal TEXT,
    status TEXT,
    created_at TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS decisions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT,
    decision TEXT,
    feedback TEXT,
    created_at TEXT
)
""")

conn.commit()


# ============================================================
# MODULE 1 — USER PROFILE
# ============================================================

def create_profile():
    print("\n" + "=" * 50)
    print("MODULE 1: CREATE USER PROFILE")
    print("=" * 50)

    name = input("Name: ")
    interests = input("Interests: ")
    skills = input("Current skills: ")
    career_goal = input("Career goal: ")
    learning_style = input(
        "Learning style (videos/books/practical/etc.): "
    )
    preferences = input("Other preferences: ")

    cursor.execute("DELETE FROM profile")

    cursor.execute("""
    INSERT INTO profile
    (name, interests, skills, career_goal,
     learning_style, preferences)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        name,
        interests,
        skills,
        career_goal,
        learning_style,
        preferences
    ))

    conn.commit()

    print("\n✅ Profile created successfully!")


def get_profile():

    cursor.execute("""
    SELECT name, interests, skills, career_goal,
           learning_style, preferences
    FROM profile
    LIMIT 1
    """)

    row = cursor.fetchone()

    if not row:
        return {}

    return {
        "name": row[0],
        "interests": row[1],
        "skills": row[2],
        "career_goal": row[3],
        "learning_style": row[4],
        "preferences": row[5]
    }


# ============================================================
# MODULE 2 — AI MEMORY
# ============================================================

def add_memory():

    print("\n" + "=" * 50)
    print("MODULE 2: AI MEMORY")
    print("=" * 50)

    memory = input(
        "What should the Digital Twin remember? "
    )

    category = input(
        "Category (preference/goal/skill/habit/other): "
    )

    cursor.execute("""
    INSERT INTO memories
    (memory, category, created_at)
    VALUES (?, ?, ?)
    """, (
        memory,
        category,
        datetime.now().isoformat()
    ))

    conn.commit()

    print("✅ Memory stored.")


def get_memories():

    cursor.execute("""
    SELECT memory, category
    FROM memories
    ORDER BY id DESC
    LIMIT 20
    """)

    rows = cursor.fetchall()

    return [
        {
            "memory": row[0],
            "category": row[1]
        }
        for row in rows
    ]


# ============================================================
# MODULE 3 — PERSONAL AI BRAIN
# ============================================================

def build_personal_context():
    """Create a safe text context from the user's profile and memories."""

    profile = get_profile()
    memories = get_memories()

    if not profile:
        profile = {
            "name": "Unknown",
            "interests": "Not provided",
            "skills": "Not provided",
            "career_goal": "Not provided",
            "learning_style": "Not provided",
            "preferences": "Not provided"
        }

    context = f"""
USER PROFILE
------------
Name: {profile.get("name", "Unknown")}
Interests: {profile.get("interests", "Not provided")}
Skills: {profile.get("skills", "Not provided")}
Career Goal: {profile.get("career_goal", "Not provided")}
Learning Style: {profile.get("learning_style", "Not provided")}
Preferences: {profile.get("preferences", "Not provided")}

USER MEMORIES
------------
"""

    if memories:
        for i, memory in enumerate(memories, start=1):
            context += (
                f"{i}. "
                f"[{memory.get('category', 'other')}] "
                f"{memory.get('memory', '')}\n"
            )
    else:
        context += "No memories stored yet.\n"

    return context


def ask_digital_twin():

    print("\n" + "=" * 50)
    print("MODULE 3: PERSONAL AI BRAIN")
    print("=" * 50)

    question = input("\nAsk your Digital Twin: ").strip()

    if not question:
        print("❌ Please enter a question.")
        return

    context = build_personal_context()

    prompt = f"""
You are an AI Digital Twin assistant.

You are NOT the real person.
You are an AI system that provides personalized
recommendations using information the user has
chosen to provide.

IMPORTANT RULES:
- Do not invent personal information.
- Use the profile and memories provided below.
- If information is missing, clearly say so.
- Give a useful and personalized answer.
- Explain briefly why the recommendation fits the user.

USER CONTEXT:
{context}

USER QUESTION:
{question}

Provide the answer in a clear and simple format.
"""

    try:

        # Gemini API call
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )

        # Check whether Gemini returned text
        if response is None:
            print("❌ Gemini returned no response.")
            return

        answer = response.text

        if not answer:
            print("❌ Gemini returned an empty response.")
            return

        print("\n🤖 DIGITAL TWIN")
        print("-" * 50)
        print(answer)

    except Exception as e:

        print("\n❌ ERROR IN MODULE 3")
        print("-" * 50)
        print(str(e))


# ============================================================
# MODULE 4 — DECISION SIMULATOR
# ============================================================

def decision_simulator():

    print("\n" + "=" * 50)
    print("MODULE 4: DECISION SIMULATOR")
    print("=" * 50)

    question = input(
        "\nWhat decision are you trying to make?\n> "
    )

    option_a = input("Option A: ")
    option_b = input("Option B: ")

    context = build_personal_context()

    prompt = f"""
You are the decision simulation engine of an AI Digital Twin.

The system knows the following about the user:

{context}

The user needs to decide between:

OPTION A:
{option_a}

OPTION B:
{option_b}

QUESTION:
{question}

Analyze both options based on the user's known:

- interests
- skills
- career goals
- learning style
- preferences
- memories

Return:

1. Option A compatibility score from 0-100
2. Option B compatibility score from 0-100
3. Advantages of A
4. Advantages of B
5. Risks of A
6. Risks of B
7. Which option currently appears more compatible
8. Explanation

Do not claim that the prediction is certain.
"""

    result = ask_ai(prompt)

    print("\n🔮 DECISION SIMULATION:")
    print(result)

    # Store decision
    cursor.execute("""
    INSERT INTO decisions
    (question, decision, feedback, created_at)
    VALUES (?, ?, ?, ?)
    """, (
        question,
        result,
        "",
        datetime.now().isoformat()
    ))

    conn.commit()


# ============================================================
# MODULE 5 — GOAL AGENT
# ============================================================

def create_goal():

    print("\n" + "=" * 50)
    print("MODULE 5: GOAL AGENT")
    print("=" * 50)

    goal = input("\nEnter your goal:\n> ")

    context = build_personal_context()

    prompt = f"""
You are the Goal Planning Agent of an AI Digital Twin.

USER INFORMATION:
{context}

USER GOAL:
{goal}

Create a personalized roadmap.

Include:

1. Current situation
2. Skills required
3. Missing skills
4. 30-day plan
5. 60-day plan
6. 90-day plan
7. Weekly activities
8. Measurable milestones
9. Possible difficulties
10. How the Digital Twin can monitor progress

Keep the plan realistic and personalized.
"""

    result = ask_ai(prompt)

    print("\n🎯 PERSONALIZED GOAL PLAN:")
    print(result)

    cursor.execute("""
    INSERT INTO goals
    (goal, status, created_at)
    VALUES (?, ?, ?)
    """, (
        goal,
        "Active",
        datetime.now().isoformat()
    ))

    conn.commit()


# ============================================================
# MODULE 6 — TWIN EVOLUTION / FEEDBACK
# ============================================================

def give_feedback():

    print("\n" + "=" * 50)
    print("MODULE 6: TWIN EVOLUTION")
    print("=" * 50)

    feedback = input(
        "\nTell the Digital Twin something it got wrong "
        "or something it should learn:\n> "
    )

    prompt = f"""
You are the memory-learning component of an AI Digital Twin.

User feedback:

{feedback}

Extract the important long-term preference or behavior.

Return ONLY JSON in this format:

{{
    "memory": "important information",
    "category": "preference/goal/skill/habit/other"
}}

If there is no useful long-term information,
return:

{{
    "memory": "",
    "category": "other"
}}
"""

    result = ask_ai(prompt)

    try:

        # Remove markdown code fences if Gemini adds them
        result = result.replace("```json", "")
        result = result.replace("```", "")
        result = result.strip()

        data = json.loads(result)

        memory = data.get("memory", "")
        category = data.get("category", "other")

        if memory:

            cursor.execute("""
            INSERT INTO memories
            (memory, category, created_at)
            VALUES (?, ?, ?)
            """, (
                memory,
                category,
                datetime.now().isoformat()
            ))

            conn.commit()

            print("\n🧠 Twin updated!")
            print("New memory:", memory)

        else:
            print("\nNo long-term memory was extracted.")

    except Exception as e:

        print("\nCould not automatically update memory.")
        print("AI response:")
        print(result)


# ============================================================
# VIEW PROFILE
# ============================================================

def view_profile():

    print("\n" + "=" * 50)
    print("CURRENT DIGITAL TWIN")
    print("=" * 50)

    profile = get_profile()

    if not profile:
        print("No profile found.")
        return

    for key, value in profile.items():
        print(f"{key}: {value}")

    print("\n🧠 MEMORIES")

    memories = get_memories()

    if not memories:
        print("No memories stored.")

    for memory in memories:
        print(
            f"- [{memory['category']}] "
            f"{memory['memory']}"
        )


# ============================================================
# MAIN APPLICATION
# ============================================================

def main():

    while True:

        print("\n")
        print("=" * 65)
        print("             AI DIGITAL TWIN CONTROL PANEL")
        print("=" * 65)

        print("""
1. Create / Update Profile
2. Add Memory
3. Ask Digital Twin
4. Decision Simulator
5. Create Goal & Roadmap
6. Teach Twin / Give Feedback
7. View Digital Twin
8. Exit
""")

        choice = input("Choose an option: ")

        if choice == "1":
            create_profile()

        elif choice == "2":
            add_memory()

        elif choice == "3":
            ask_digital_twin()

        elif choice == "4":
            decision_simulator()

        elif choice == "5":
            create_goal()

        elif choice == "6":
            give_feedback()

        elif choice == "7":
            view_profile()

        elif choice == "8":
            print("\n👋 Digital Twin closed.")
            break

        else:
            print("\n❌ Invalid option.")


# ============================================================
# START
# ============================================================

main()

          AI DIGITAL TWIN OF A PERSON
Enter your Gemini API Key: ··········


             AI DIGITAL TWIN CONTROL PANEL

1. Create / Update Profile
2. Add Memory
3. Ask Digital Twin
4. Decision Simulator
5. Create Goal & Roadmap
6. Teach Twin / Give Feedback
7. View Digital Twin
8. Exit

Choose an option: 3

MODULE 3: PERSONAL AI BRAIN

Ask your Digital Twin: Based on my interests, skills, preferences and career goal, what should I focus on learning next?

🤖 DIGITAL TWIN
--------------------------------------------------
Based on the profile and memories currently provided, I do not have any information about your interests, skills, preferences, or career goals. 

Therefore, I cannot provide a personalized recommendation at this time. 

**What you can do next:**
Please share your interests, current skills, preferences, and career goals with me. Once you provide that information, I will be able to give you a tailored learning recommendation and explain why it fits your background!


  